In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q dagshub mlflow optuna

import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.model_selection import TimeSeriesSplit

import dagshub
import mlflow
import mlflow.lightgbm

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_columns', 50)

In [ ]:
DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"
OUTPUT_DIR = "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
VAL_HOLDOUT_WEEKS = 13   
N_HPO_TRIALS = 20
CHRISTMAS_SHIFT_FRACTION = 1 / 7

In [ ]:
dagshub.init(repo_owner='lshek22', repo_name='walmart-recruiting-store-sales-forecasting', mlflow=True)
mlflow.set_experiment("LightGBM_v3_Training")

In [ ]:
train_raw = pd.read_csv(os.path.join(DATA_DIR, "train.csv.zip"), parse_dates=["Date"])
test_raw = pd.read_csv(os.path.join(DATA_DIR, "test.csv.zip"), parse_dates=["Date"])
stores = pd.read_csv(os.path.join(DATA_DIR, "stores.csv"))
features = pd.read_csv(os.path.join(DATA_DIR, "features.csv.zip"), parse_dates=["Date"])
features["IsHoliday"] = features["IsHoliday"].astype(bool)

print("train:", train_raw.shape, "test:", test_raw.shape, "stores:", stores.shape, "features:", features.shape)
train_raw.head()


In [ ]:
def clean_features(features_df):
    df = features_df.copy()
    markdown_cols = [c for c in df.columns if c.startswith("MarkDown")]

    n_missing_before = int(df[markdown_cols].isna().sum().sum())
    n_negative = int((df[markdown_cols] < 0).sum().sum())
    df[markdown_cols] = df[markdown_cols].fillna(0).clip(lower=0)

    for col in ["CPI", "Unemployment"]:
        df[col] = df.groupby("Store")[col].transform(lambda s: s.ffill().bfill())

    stats = {
        "markdown_missing_before": n_missing_before,
        "markdown_negative_values_clipped": n_negative,
        "markdown_missing_after": int(df[markdown_cols].isna().sum().sum()),
        "cpi_unemployment_missing_after": int(df[["CPI", "Unemployment"]].isna().sum().sum()),
        "n_rows": len(df),
    }
    return df, stats


mlflow.set_experiment("LightGBM_v3_Cleaning")
with mlflow.start_run(run_name="clean_features"):
    features_clean, clean_stats = clean_features(features)
    mlflow.log_params({
        "markdown_fill_strategy": "fillna_0_clip_negative",
        "cpi_unemployment_fill_strategy": "groupby_store_ffill_bfill",
    })
    mlflow.log_metrics(clean_stats)

print(clean_stats)

In [ ]:
HOLIDAY_DATES = {
    "SuperBowl": ["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"],
    "LaborDay": ["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"],
    "Thanksgiving": ["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"],
    "Christmas": ["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"],
}
HOLIDAY_FLAG_COLS = [f"Is{name}" for name in HOLIDAY_DATES]


def merge_frame(df, stores_df, features_df):
    out = df.merge(stores_df, on="Store", how="left")
    out = out.merge(features_df.drop(columns=["IsHoliday"]), on=["Store", "Date"], how="left")
    return out


def engineer_features(train_df, test_df, stores_df, features_df):
    train_m = merge_frame(train_df, stores_df, features_df)
    test_m = merge_frame(test_df, stores_df, features_df)
    test_m["Weekly_Sales"] = np.nan

    full = pd.concat([train_m, test_m], axis=0, ignore_index=True, sort=False)
    full = full.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

    for name, dates in HOLIDAY_DATES.items():
        full[f"Is{name}"] = full["Date"].isin(pd.to_datetime(dates))
    full["PreHolidayWeek"] = (
        full.groupby(["Store", "Dept"])[HOLIDAY_FLAG_COLS]
        .transform(lambda s: s.shift(-1))
        .any(axis=1)
        .fillna(False)
    )

    full["Year"] = full["Date"].dt.year
    full["Month"] = full["Date"].dt.month
    full["WeekOfYear"] = full["Date"].dt.isocalendar().week.astype(int)
    full["IsHoliday"] = full["IsHoliday"].astype(bool)

    g = full.groupby(["Store", "Dept"])["Weekly_Sales"]
    full["lag_1"] = g.shift(1)
    full["lag_2"] = g.shift(2)
    full["lag_4"] = g.shift(4)
    full["lag_8"] = g.shift(8)
    full["lag_52"] = g.shift(52)
    full["rolling_mean_4"] = full.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
        lambda s: s.shift(1).rolling(4, min_periods=1).mean())
    full["rolling_mean_8"] = full.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
        lambda s: s.shift(1).rolling(8, min_periods=1).mean())
    full["rolling_std_4"] = full.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
        lambda s: s.shift(1).rolling(4, min_periods=1).std())

    # Leakage-free (shifted, expanding) target encodings
    full["storedept_avg_sales"] = full.groupby(["Store", "Dept"])["Weekly_Sales"].transform(
        lambda s: s.expanding().mean().shift(1))

    full = full.sort_values(["Dept", "WeekOfYear", "Year"]).reset_index(drop=True)
    full["dept_woy_avg_sales"] = full.groupby(["Dept", "WeekOfYear"])["Weekly_Sales"].transform(
        lambda s: s.expanding().mean().shift(1))
    full = full.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

    engineered_num_cols = ["lag_1", "lag_2", "lag_4", "lag_8", "lag_52", "rolling_mean_4",
                            "rolling_mean_8", "rolling_std_4", "storedept_avg_sales", "dept_woy_avg_sales"]
    n_missing_before = int(full[engineered_num_cols].isna().sum().sum())
    for c in engineered_num_cols:
        full[c] = full.groupby(["Store", "Dept"])[c].transform(lambda s: s.fillna(s.median()))
        full[c] = full[c].fillna(full[c].median())
    n_missing_after = int(full[engineered_num_cols].isna().sum().sum())

    stats = {
        "n_rows_total": len(full),
        "n_engineered_missing_before_fill": n_missing_before,
        "n_engineered_missing_after_fill": n_missing_after,
    }
    return full, engineered_num_cols, stats


mlflow.set_experiment("LightGBM_v3_Feature_Engineering")
with mlflow.start_run(run_name="engineer_features"):
    full, engineered_num_cols, fe_stats = engineer_features(train_raw, test_raw, stores, features_clean)
    mlflow.log_params({
        "lag_features": "lag_1,lag_2,lag_4,lag_8,lag_52",
        "rolling_features": "rolling_mean_4,rolling_mean_8,rolling_std_4",
        "target_encodings": "storedept_avg_sales (expanding, shifted), dept_woy_avg_sales (expanding, shifted)",
        "holiday_dummies": ",".join(HOLIDAY_FLAG_COLS) + ",PreHolidayWeek",
    })
    mlflow.log_metrics(fe_stats)

print(fe_stats)
full.head()

In [ ]:
CATEGORICAL_FEATURES = ["Store", "Dept", "Type"] + HOLIDAY_FLAG_COLS + ["IsHoliday", "PreHolidayWeek"]
for c in CATEGORICAL_FEATURES:
    full[c] = full[c].astype("category")

CANDIDATE_FEATURES = (
    ["Store", "Dept", "Type", "Size", "Temperature", "Fuel_Price",
     "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
     "CPI", "Unemployment", "IsHoliday", "Year", "Month", "WeekOfYear"]
    + HOLIDAY_FLAG_COLS + ["PreHolidayWeek"] + engineered_num_cols
)

train_f = full[full["Weekly_Sales"].notna()].copy().sort_values("Date").reset_index(drop=True)
test_f = full[full["Weekly_Sales"].isna()].copy()


def select_features(train_df, candidate_features, categorical_features, random_state=42):
    X = train_df[candidate_features]
    y = train_df["Weekly_Sales"]
    w = np.where(train_df["IsHoliday"], 5, 1)

    probe_model = lgb.LGBMRegressor(objective="regression_l1", n_estimators=150,
                                     random_state=random_state, verbosity=-1)
    probe_model.fit(X, y, sample_weight=w, categorical_feature=categorical_features)

    importances = pd.Series(probe_model.feature_importances_, index=candidate_features).sort_values(ascending=False)
    selected = importances[importances > 0].index.tolist()
    return selected, importances


mlflow.set_experiment("LightGBM_v3_Feature_Selection")
with mlflow.start_run(run_name="select_features"):
    selected_features, importances = select_features(train_f, CANDIDATE_FEATURES, CATEGORICAL_FEATURES, RANDOM_STATE)

    mlflow.log_param("n_candidate_features", len(CANDIDATE_FEATURES))
    mlflow.log_param("n_selected_features", len(selected_features))
    mlflow.log_param("selected_features", ",".join(selected_features))
    mlflow.log_metrics({f"importance_{k}": float(v) for k, v in importances.items()})

    importance_path = os.path.join(OUTPUT_DIR, "feature_importances_v3.csv")
    importances.to_csv(importance_path, header=["importance"])
    mlflow.log_artifact(importance_path)

selected_cat_features = [c for c in CATEGORICAL_FEATURES if c in selected_features]
print("Selected features:", selected_features)
importances

In [ ]:
cutoff = train_f["Date"].max() - pd.Timedelta(weeks=VAL_HOLDOUT_WEEKS)
tr_mask = train_f["Date"] <= cutoff
val_mask = train_f["Date"] > cutoff

X_tr, y_tr = train_f.loc[tr_mask, selected_features], train_f.loc[tr_mask, "Weekly_Sales"]
w_tr = np.where(train_f.loc[tr_mask, "IsHoliday"], 5, 1)
X_val, y_val = train_f.loc[val_mask, selected_features], train_f.loc[val_mask, "Weekly_Sales"]
val_is_holiday = train_f.loc[val_mask, "IsHoliday"].astype(bool).values

print(f"train rows: {tr_mask.sum()}, val rows: {val_mask.sum()}")
print(f"val date range: {train_f.loc[val_mask, 'Date'].min()} to {train_f.loc[val_mask, 'Date'].max()}")


def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))

In [ ]:
mlflow.set_experiment("LightGBM_v3_HPO")
with mlflow.start_run(run_name="optuna_search") as hpo_parent:

    def objective(trial):
        params = dict(
            objective="regression_l1",
            n_estimators=300,
            learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
            num_leaves=trial.suggest_int("num_leaves", 15, 127),
            min_child_samples=trial.suggest_int("min_child_samples", 5, 50),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            random_state=RANDOM_STATE,
            verbosity=-1,
        )
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr, sample_weight=w_tr, categorical_feature=selected_cat_features)
        preds = model.predict(X_val)
        score = wmae(y_val.values, preds, val_is_holiday)

        with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
            mlflow.log_params(params)
            mlflow.log_metric("val_wmae", score)
        return score

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_HPO_TRIALS)

    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_val_wmae", study.best_value)

best_params = dict(objective="regression_l1", n_estimators=400, random_state=RANDOM_STATE,
                    verbosity=-1, **study.best_params)
print("Best params:", study.best_params)
print("Best HPO validation WMAE:", study.best_value)

In [ ]:
mlflow.set_experiment("LightGBM_v3_Training")
with mlflow.start_run(run_name="train_and_blend"):
    mlflow.log_params(best_params)
    mlflow.log_param("val_holdout_weeks", VAL_HOLDOUT_WEEKS)
    mlflow.log_param("christmas_shift_fraction", CHRISTMAS_SHIFT_FRACTION)

    val_model = lgb.LGBMRegressor(**best_params)
    val_model.fit(X_tr, y_tr, sample_weight=w_tr, categorical_feature=selected_cat_features)
    val_preds = val_model.predict(X_val)
    val_wmae = wmae(y_val.values, val_preds, val_is_holiday)
    mlflow.log_metric("val_wmae_model_only", val_wmae)

    naive_val_preds = X_val["lag_52"].values if "lag_52" in selected_features else None
    best_alpha, best_blend_wmae = 1.0, val_wmae
    if naive_val_preds is not None:
        for alpha in np.linspace(0, 1, 21):
            blend = alpha * val_preds + (1 - alpha) * naive_val_preds
            score = wmae(y_val.values, blend, val_is_holiday)
            if score < best_blend_wmae:
                best_alpha, best_blend_wmae = alpha, score
    mlflow.log_metric("best_blend_alpha", best_alpha)
    mlflow.log_metric("val_wmae_blended", best_blend_wmae)

    w_full = np.where(train_f["IsHoliday"], 5, 1)
    final_model = lgb.LGBMRegressor(**best_params)
    final_model.fit(train_f[selected_features], train_f["Weekly_Sales"],
                     sample_weight=w_full, categorical_feature=selected_cat_features)
    mlflow.lightgbm.log_model(final_model, artifact_path="model")

    test_model_preds = final_model.predict(test_f[selected_features])
    if "lag_52" in selected_features:
        test_naive_preds = test_f["lag_52"].values
        test_blend = best_alpha * test_model_preds + (1 - best_alpha) * test_naive_preds
    else:
        test_blend = test_model_preds

    def christmas_shift_correction(df, preds, shift_fraction=CHRISTMAS_SHIFT_FRACTION):
        preds = preds.copy()
        df = df.reset_index(drop=True)
        idx_christmas = np.where(df["IsChristmas"].astype(bool).values)[0]
        for i in idx_christmas:
            store, dept, date = df.loc[i, "Store"], df.loc[i, "Dept"], df.loc[i, "Date"]
            prev_mask = (df["Store"] == store) & (df["Dept"] == dept) & (df["Date"] == date - pd.Timedelta(weeks=1))
            prev_idx = df.index[prev_mask]
            if len(prev_idx) == 1:
                shift_amt = preds[i] * shift_fraction
                preds[i] -= shift_amt
                preds[prev_idx[0]] += shift_amt
        return preds

    test_f_reset = test_f.reset_index(drop=True)
    test_final_preds = christmas_shift_correction(test_f_reset, test_blend, CHRISTMAS_SHIFT_FRACTION)

    submission = test_f_reset.copy()
    submission["Weekly_Sales"] = test_final_preds
    submission["Id"] = (
        submission["Store"].astype(int).astype(str) + "_" +
        submission["Dept"].astype(int).astype(str) + "_" +
        submission["Date"].dt.strftime("%Y-%m-%d")
    )
    submission = submission[["Id", "Weekly_Sales"]]

    submission_path = os.path.join(OUTPUT_DIR, "submission_v3.csv")
    submission.to_csv(submission_path, index=False)
    mlflow.log_artifact(submission_path)

print(f"Model-only validation WMAE: {val_wmae:.2f}")
print(f"Blended validation WMAE:    {best_blend_wmae:.2f} (alpha={best_alpha:.2f})")
print(f"Saved {len(submission)} predictions to {submission_path}")

In [ ]:
submission.head()